# Pull data

In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3

## Functions

In [2]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name='dustin-payment-analysis'):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

## Constants

In [3]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# output
str_dirname_output = './output'

Project: 20231010-gen-xii


## Output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

## Read query

In [5]:
str_filepath = './sql/query.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('with tbl1 as\n'
 '(\n'
 '\tselect\n'
 '\t\tintAccountKey as bigAccountId,\n'
 '\t\tfltDebtor_Score_pd,\n'
 '\t\tfltDebtor_Score_lgd,\n'
 '\t\tfltDebtorScore,\n'
 '\t\tstrScoreCardVersion, \n'
 '\t\tdtmStampCreation,\n'
 '\t\tRow_number() OVER(partition BY intAccountKey, strScoreCardVersion ORDER '
 'BY dtmStampCreation desc) RowNum\n'
 '\tfrom edw.pfsedw.dbo.DimScoreCard\n'
 "\twhere strScoreCardVersion in ('genxi_v2_2', 'genxi_v2_3')\n"
 "\tand dtmStampCreation >= '2024-03-01'\n"
 ')\n'
 'select*\n'
 'from tbl1\n'
 'where RowNum = 1\n'
 '\n'
 '\n'
 '\n')


## Write into df

In [6]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()
# info
print(f'Data contains {df.shape[0]} rows and {df.shape[1]} columns')

Data contains 97705 rows and 7 columns
Wall time: 1.6 s


In [7]:
# preview
df.head()

,bigAccountId,fltDebtor_Score_pd,fltDebtor_Score_lgd,fltDebtorScore,strScoreCardVersion,dtmStampCreation,RowNum
0,7034142,0.091244,0.315624,0.023507,genxi_v2_2,2024-03-18 18:45:41.733,1
1,7064762,0.148281,0.374718,0.161798,genxi_v2_2,2024-03-18 18:46:48.540,1
2,7074217,0.176167,0.459325,0.153078,genxi_v2_2,2024-03-18 18:46:01.620,1
3,7078993,0.185790,0.386467,0.107600,genxi_v2_2,2024-03-18 18:45:55.240,1
4,7080303,0.242972,0.508737,0.208911,genxi_v2_2,2024-03-18 18:46:02.467,1


In [8]:
for col in df.columns:
    print(col)

bigAccountId
fltDebtor_Score_pd
fltDebtor_Score_lgd
fltDebtorScore
strScoreCardVersion
dtmStampCreation
RowNum


In [9]:
# get prop nan
df.isnull().mean()

bigAccountId           0.0
fltDebtor_Score_pd     0.0
fltDebtor_Score_lgd    0.0
fltDebtorScore         0.0
strScoreCardVersion    0.0
dtmStampCreation       0.0
RowNum                 0.0
dtype: float64

### Save

In [10]:
%%time

# save
str_filename = 'df_genxi_with_factors.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_csv(str_local_path, index=False)

Wall time: 1.33 s


## Upload to s3

In [11]:
# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'ad_hoc/pull_gen_11_scores/{str_filename}', 
    str_bucket_name=str_project,
)

## Clean-up

In [12]:
os.remove(str_local_path)